# Calibra: From Robot Demos to GR00T Coreset

End-to-end walkthrough showing how to **audit**, **certify**, **prune**, and **retarget** a robot manipulation dataset before NVIDIA GR00T N1.7+ fine-tuning.

**What you'll do:**
1. Build a synthetic Isaac Lab–style dataset (50 episodes, 50 Hz, 7-DOF)
2. Inject realistic corruptions — jerk spikes, timestamp dropout, missing language annotations
3. Run the Calibra diagnostic pipeline and catch every injected defect
4. Run GR00T-specific structural checks
5. Prune to a high-quality, diverse coreset (50 → 25 episodes)
6. Retarget absolute EEF poses to the relative 6-DOF deltas GR00T N1.7+ expects
7. Wire `calibra certify` into a CI quality gate

**Runtime:** ~10 seconds on CPU. No GPU, no actual dataset downloads required.

---

## Setup

In [ ]:
# Install Calibra with kinematics support (scipy — needed for the retarget step)
# !pip install 'calibra[kinematics]'

In [ ]:
import numpy as np

from calibra.schema.episode import Episode, EpisodeBatch, EpisodeMetadata
from calibra.pipeline import Pipeline
from calibra.pruning import CoresetSelector
from calibra.kinematics.retarget import retarget_episode_eef

---
## 1. Build a Synthetic Isaac Lab Dataset

We create 50 episodes at **50 Hz** with **7-DOF actions** (6-DOF arm + binary gripper), plus EEF pose observations matching the Isaac Lab HDF5 format.

Injected defects (matches real-world Isaac Sim issues):
- **Episodes 0–7**: jerk spikes — large random action jumps, e.g. from controller glitches
- **Episodes 0–2**: timestamp dropout — duplicated timestamps from dropped physics steps
- **Episodes 48–49**: missing language annotations — task string not set in episode metadata

In [ ]:
rng = np.random.default_rng(42)

DT = 1 / 50  # 50 Hz control loop
N_STEPS = 200  # 4-second episodes

TASKS = [
    "pick up the red cube and place it in the bin",
    "grasp the bottle and move it to the target zone",
    "push the block to the left marker",
    "lift the object and place it on the shelf",
]


def make_episode(
    episode_id: str,
    inject_spikes: bool = False,
    inject_dropout: bool = False,
    missing_language: bool = False,
) -> Episode:
    """Generate one synthetic manipulation episode."""
    t = np.arange(N_STEPS, dtype=np.float64) * DT

    # Smooth 6-DOF arm trajectory — sum of sinusoids, realistic manipulation motion
    freqs = rng.uniform(0.1, 0.8, 6)
    phases = rng.uniform(0.0, 2 * np.pi, 6)
    amps = rng.uniform(0.05, 0.3, 6)
    arm = np.stack(
        [a * np.sin(2 * np.pi * f * t + p) for a, f, p in zip(amps, freqs, phases)],
        axis=1,
    )  # (T, 6)

    # Binary gripper: open for first half, closed for second half
    gripper = np.where(t > t[N_STEPS // 2], 1.0, -1.0).reshape(-1, 1)
    actions = np.concatenate([arm, gripper], axis=1).astype(np.float32)  # (T, 7)

    # EEF world-frame position — integrate arm deltas
    eef_pos = np.cumsum(arm * DT, axis=0).astype(np.float64)
    eef_pos += rng.uniform(-0.1, 0.1, (1, 3))  # random start position

    # EEF quaternion [qx, qy, qz, qw] — slowly rotating, starting near identity
    raw_q = np.tile([0.0, 0.0, 0.0, 1.0], (N_STEPS, 1))
    raw_q += rng.normal(0, 0.02, (N_STEPS, 4))
    eef_quat = raw_q / np.linalg.norm(raw_q, axis=1, keepdims=True)

    # Timestamps: uniform 50 Hz with realistic sub-millisecond jitter
    timestamps = t + rng.normal(0, 5e-5, N_STEPS)
    timestamps[0] = t[0]
    timestamps = np.sort(timestamps)  # ensure monotonic

    # --- inject defects ---

    if inject_spikes:
        # Simulate controller glitch: 20 steps with large random action jumps
        idx = rng.choice(N_STEPS - 2, size=20, replace=False) + 1
        actions[idx] += rng.choice([-1, 1], size=(20, 7)).astype(np.float32) * rng.uniform(
            1.5, 3.0, (20, 7)
        ).astype(np.float32)

    if inject_dropout:
        # Simulate dropped physics steps: duplicate 15 timestamps
        drop_idx = rng.choice(range(5, N_STEPS - 5), size=15, replace=False)
        timestamps[drop_idx] = timestamps[drop_idx - 1]

    return Episode(
        metadata=EpisodeMetadata(
            episode_id=episode_id,
            task_description=None if missing_language else str(rng.choice(TASKS)),
            success=not inject_spikes,
        ),
        timestamps=timestamps,
        observations={
            "eef_pos": eef_pos,  # (T, 3)  world-frame position
            "eef_quat": eef_quat,  # (T, 4)  quaternion [qx, qy, qz, qw]
            "proprio": arm.copy(),  # (T, 6)  joint positions
            # Placeholder camera tensor — GR00T check only looks at the key name
            "camera_rgb": np.zeros((N_STEPS, 4, 4, 3), dtype=np.uint8),
        },
        actions=actions,
    )

In [ ]:
episodes = [
    make_episode(
        episode_id=f"ep_{i:03d}",
        inject_spikes=(i < 8),
        inject_dropout=(i < 3),
        missing_language=(i >= 48),
    )
    for i in range(50)
]

batch = EpisodeBatch(
    episodes=episodes,
    dataset_name="isaac_lab_synthetic",
    format="hdf5",
    source_path="/data/isaac_lab_demos.h5",
)

print(f"Episodes : {batch.n_episodes}")
print(f"Steps    : {batch.n_samples:,}  ({batch.n_samples / 50:.0f} s total)")
print(f"Modalities: {sorted(batch.modalities)}")
print(f"Action dim: {batch.episodes[0].action_dim}D")

---
## 2. Run the Calibra Audit

`Pipeline().run()` runs four analyzers over every episode and returns a `DiagnosticReport` with aggregate flags, 95% bootstrap confidence intervals, and per-episode metric arrays.

| Analyzer | What it measures |
|---|---|
| `TemporalAnalyzer` | Timestamp jitter CV, dropout rate, camera–proprioception lag |
| `ControlSmoothnessAnalyzer` | LDLJ, jerk spike rate, velocity discontinuity rate |
| `CoverageEntropyAnalyzer` | Action entropy (bits/dim), PCA variance, episode length distribution |
| `TaskStructureAnalyzer` | Contact density, grasp events/episode, trajectory diversity |

In [ ]:
report = Pipeline().run(batch)
print(report.summary())

### Per-episode outlier detection

In addition to aggregate flags, Calibra uses **MAD (Median Absolute Deviation)** to identify which specific episodes are responsible for the anomalies. MAD is robust to the heavy-tailed distributions typical of manipulation data.

In [ ]:
from calibra.anomalies import find_outliers, render

outliers = find_outliers(report)
if outliers:
    print(render(outliers, report.n_episodes))
else:
    print("No per-episode outliers detected.")

---
## 3. GR00T Compatibility Check

Pass `policy_family="gr00t"` to activate five GR00T-specific structural checks:

| Check | GR00T requirement |
|---|---|
| **Visual modality** | At least one RGB camera stream per episode |
| **Language annotations** | Task instruction string on every episode |
| **Episode length** | All episodes ≥ 16 steps (default action chunk size) |
| **Control frequency** | 15–120 Hz |
| **Action dimension** | 7D, 8D, 14D, or 16D (documented GR00T robot configs) |

Our dataset has two episodes with missing language annotations — the check will surface this.

In [ ]:
gr00t_report = Pipeline().run(batch, policy_family="gr00t")
print(gr00t_report.summary())

In [ ]:
# Inspect GR00T-specific flags directly
gr00t_results = [
    r for r in gr00t_report.analyzer_results if r.analyzer_name == "gr00t_compatibility"
]
if gr00t_results:
    for flag in gr00t_results[0].flags:
        icon = {"OK": "✓", "WARNING": "⚠", "CRITICAL": "✗", "INFO": "·"}.get(flag.level.value, "?")
        print(f"{icon}  [{flag.level.value:<8}]  {flag.metric}")
        print(f"          {flag.interpretation}")
        if flag.implication and flag.level.value in ("WARNING", "CRITICAL"):
            print(f"          → {flag.implication}")
        print()

---
## 4. Prune to Coreset

Two-stage pipeline that reduces 50 episodes to a smaller, cleaner, more diverse set:

**Stage 1 — Quality filter**: removes episodes that exceed jerk spike, velocity discontinuity, or timestamp dropout thresholds. Our 8 injected spike episodes should all be removed here.

**Stage 2 — Greedy max-coverage**: from the quality-passing pool, selects K episodes using **farthest-point sampling** on action-space statistics — ensuring the coreset covers as much of the behavioral distribution as possible. O(N × K) — handles ~50k episodes without approximation.

`entropy_weight=0.4` biases selection toward **high-entropy (informationally rich) episodes**, which improves GR00T fine-tuning generalization.

In [ ]:
selector = CoresetSelector(
    keep_fraction=0.5,  # keep 50% of episodes
    entropy_weight=0.4,  # bias toward informationally rich episodes
)
result = selector.select(batch, report)
print(result.summary())

In [ ]:
# Build the coreset EpisodeBatch from the selected IDs
keep_ids = set(result.keep_episode_ids)

coreset = EpisodeBatch(
    episodes=[ep for ep in batch.episodes if ep.metadata.episode_id in keep_ids],
    dataset_name=batch.dataset_name + "_coreset",
    format=batch.format,
    source_path=batch.source_path,
)

print(f"Original  : {batch.n_episodes} episodes")
print(f"Coreset   : {coreset.n_episodes} episodes")
print(f"Quality failures removed : {len(result.quality_fail_ids)}  {result.quality_fail_ids}")
print(f"Diversity-pruned removed : {len(result.diversity_pruned_ids)}")

# Verify no corrupted episodes survived into the coreset
corrupted_in_coreset = [
    ep.metadata.episode_id for ep in coreset.episodes if not ep.metadata.success
]
print(f"Corrupted episodes in coreset: {corrupted_in_coreset or 'none'}")

---
## 5. Retarget: Absolute EEF → Relative Deltas

**GR00T N1.7+ uses a Relative End-Effector (EEF) action space.** Isaac Lab records actions in world-frame absolute coordinates. Passing absolute poses directly to GR00T breaks training: the policy learns to predict world-frame displacements from an arbitrary origin, which do not generalize across episodes or embodiments.

**Mathematical formulation:**

Given absolute EEF translation $p_t \in \mathbb{R}^3$ and rotation $R_t \in SO(3)$:

$$\Delta p_t = R_t^\top (p_{t+1} - p_t) \quad \text{(position delta in current EEF frame)}$$

$$\Delta R_t = R_t^\top R_{t+1} \to \text{Euler}_{xyz} \quad \text{(rotation delta as roll-pitch-yaw, radians)}$$

Output shape: `(T−1, 6)` — one fewer step than input. Use `--pad` to append a zero row for fixed-length policies.

In [ ]:
retargeted_actions = []

for ep in coreset.episodes:
    rel = retarget_episode_eef(
        eef_pos=ep.observations["eef_pos"],  # (T, 3)
        eef_quat=ep.observations["eef_quat"],  # (T, 4) [qx, qy, qz, qw]
    )  # returns (T-1, 6)
    retargeted_actions.append(rel)

all_deltas = np.concatenate(retargeted_actions, axis=0)

print(f"Input  action shape : {coreset.episodes[0].actions.shape}  (7-DOF world-frame)")
print(f"Output action shape : {retargeted_actions[0].shape}  [dx, dy, dz, droll, dpitch, dyaw]")
print()
print("Delta magnitudes (mean ± std across all coreset steps):")
print(f"  Position : {np.abs(all_deltas[:, :3]).mean():.5f} ± {all_deltas[:, :3].std():.5f} m/step")
print(
    f"  Rotation : {np.abs(all_deltas[:, 3:]).mean():.5f} ± {all_deltas[:, 3:].std():.5f} rad/step"
)

In [ ]:
# Optional: save one episode to .npz (matches calibra retarget CLI output format)
import tempfile
import os

out_dir = tempfile.mkdtemp(prefix="calibra_retargeted_")

for ep, rel in zip(coreset.episodes, retargeted_actions):
    out_path = os.path.join(out_dir, f"{ep.metadata.episode_id}.npz")
    np.savez_compressed(
        out_path,
        relative_actions=rel,
        episode_id=np.bytes_(ep.metadata.episode_id),
        source_path=np.bytes_(batch.source_path),
    )

print(f"Saved {coreset.n_episodes} .npz files to {out_dir}")

# Verify round-trip
sample = np.load(os.path.join(out_dir, f"{coreset.episodes[0].metadata.episode_id}.npz"))
print(
    f"  relative_actions : shape={sample['relative_actions'].shape}, dtype={sample['relative_actions'].dtype}"
)
print(f"  episode_id       : {sample['episode_id']}")

---
## 6. Wire into CI

`calibra certify` exits with a structured status code — add it as a quality gate in your data collection pipeline.

```
Exit 0 — CERTIFIED             (all checks pass)
Exit 1 — PROVISIONALLY CERTIFIED  (warnings only)
Exit 2 — NOT CERTIFIED          (critical failures)
```

**CLI (simplest):**
```bash
# Fails the pipeline if data has critical issues
calibra certify /data/isaac_lab_demos.h5 --policy gr00t --strict

# Machine-readable JSON for downstream scripting
calibra certify /data/isaac_lab_demos.h5 --policy gr00t --json > report.json
```

**Python API:**

In [ ]:
from calibra.schema.report import RiskLevel


def certify_for_gr00t(batch: EpisodeBatch, strict: bool = False) -> int:
    """
    Run GR00T certification on a batch.
    Returns exit code: 0 = certified, 1 = warnings, 2 = critical failures.
    """
    report = Pipeline().run(batch, policy_family="gr00t")

    criticals = report.flags_at_level(RiskLevel.CRITICAL)
    warnings = report.flags_at_level(RiskLevel.WARNING)

    if criticals:
        print("NOT CERTIFIED — critical issues must be resolved before GR00T fine-tuning:")
        for f in criticals:
            print(f"  ✗ {f.metric}: {f.interpretation}")
        return 2

    if warnings:
        print("PROVISIONALLY CERTIFIED — warnings (may reduce fine-tuning quality):")
        for f in warnings:
            print(f"  ⚠ {f.metric}: {f.interpretation}")
        return 1 if strict else 0

    print("CERTIFIED — dataset passes all GR00T N1 structural checks.")
    return 0


# On the original (dirty) batch
print("=== Original dataset ===")
code = certify_for_gr00t(batch)
print(f"Exit code: {code}")
print()

# On the coreset (quality-filtered, language issues may remain in eps 48-49)
print("=== Coreset ===")
code = certify_for_gr00t(coreset)
print(f"Exit code: {code}")

---
## Summary

| Step | Command | What it does |
|---|---|---|
| Audit | `Pipeline().run(batch)` | Full diagnostic report: jitter, LDLJ, entropy, contact density |
| GR00T check | `Pipeline().run(batch, policy_family="gr00t")` | Visual obs, language, frequency, action dim |
| Prune | `CoresetSelector(keep_fraction=0.5).select(batch, report)` | Quality filter + greedy max-coverage coreset |
| Retarget | `retarget_episode_eef(eef_pos, eef_quat)` | Absolute EEF → relative 6-DOF deltas for GR00T N1.7+ |
| CI gate | `calibra certify /data/demos.h5 --policy gr00t` | Structured exit codes for pipeline integration |

**Next steps:**
- Load a real Isaac Lab dataset: `calibra certify /path/to/isaac_lab.h5 --policy gr00t`
- Compare against a reference: `calibra compare /path/to/my_demos aloha`
- Profile a new dataset to add to the evidence base: `python scripts/profile_dataset.py lerobot/my_dataset`

See the [contributing guide](../README.md#contributing) to add your dataset as a reference profile — every new profile strengthens the evidence base for `calibra compare`.